# Semana 10 — Redes neurais para classificação tabular
## Fundamentos, treinamento e comparação com Random Forest

A aula permanece dentro do CRISP-DM e avalia uma MLP como modelo candidato para classificação de falhas industriais. O experimento usa o mesmo conjunto de teste para a MLP e para a Random Forest.

**Questão técnica:** a rede neural oferece ganho suficiente em desempenho ou custo operacional para justificar maior sensibilidade à preparação e aos hiperparâmetros?


### Resultados de aprendizagem

- descrever neurônio, camada, peso, viés, ativação e função de perda;
- explicar forward pass, backpropagation e atualização por otimizador;
- construir um pipeline sem vazamento de dados;
- treinar MLP e Random Forest sob o mesmo protocolo;
- inspecionar matrizes de pesos e reproduzir manualmente um forward pass;
- diagnosticar convergência com curvas de treino e validação;
- comparar modelos com métricas e custo de falsos positivos e negativos.


## 1. Continuidade do CRISP-DM

Não reiniciaremos o ciclo. O problema, as métricas e o pipeline já existem. A mudança desta aula ocorre principalmente em **Data Preparation** e **Modeling**:

| Decisão já conhecida | Consequência para a rede neural |
|---|---|
| Separar treino e teste | O teste continua intocado até a avaliação final |
| Imputar valores ausentes | A rede não opera corretamente com `NaN` |
| Codificar categorias | Entradas precisam ser numéricas |
| Avaliar custo do falso negativo | Define qual métrica merece prioridade |
| Criar janelas temporais | Permite representar histórico como atributos |
| Comparar com baseline | Mostra se a complexidade adicional trouxe valor |

Uma rede neural é uma candidata dentro do CRISP-DM, não uma substituta para o processo.

![Da Random Forest à rede neural](images/semana_10_random_forest_para_mlp.png)

## 2. Fundamentos da MLP

Para uma observação (x), cada camada densa calcula:

$$ z^{(l)} = a^{(l-1)}W^{(l)} + b^{(l)} $$

$$ a^{(l)} = f(z^{(l)}) $$

A camada de saída binária usa sigmoid para produzir (p(y=1|x)). Durante o treinamento, a binary cross-entropy mede o erro. O backpropagation aplica a regra da cadeia para calcular os gradientes, e o Adam atualiza pesos e vieses.

### Arquitetura usada

`4 entradas → 16 neurônios ReLU → 8 neurônios ReLU → 1 saída sigmoid`

- **época:** uma passagem pelo conjunto de treino;
- **batch:** subconjunto usado em uma atualização;
- **loss:** objetivo numérico minimizado;
- **early stopping:** interrupção quando a validação interna deixa de melhorar;
- **overfitting:** melhora no treino sem generalização equivalente.


In [ ]:
import numpy as np

# Um neurônio simples: três medições já normalizadas.
x = np.array([0.8, 1.2, -0.4])       # temperatura, vibração e corrente
w = np.array([0.7, 1.1, -0.2])
b = -0.3

z = x @ w + b
relu = max(0.0, z)
sigmoid = 1 / (1 + np.exp(-z))

print(f"Combinação linear z = {z:.3f}")
print(f"ReLU(z) = {relu:.3f}")
print(f"Sigmoid(z) = {sigmoid:.3f}")

## 3. Por que normalizar?

Temperatura pode variar em dezenas de graus, vibração em poucos milímetros por segundo e corrente em centenas de ampères. Sem padronização, uma variável de grande magnitude pode dominar o processo de otimização.

O `StandardScaler` calcula média e desvio **somente no treino**:

$$ z = \frac{x-\mu_{treino}}{\sigma_{treino}} $$

Aplicar `fit` antes da separação causa **vazamento de dados**: informações do teste participam indiretamente do treinamento.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 900
temperatura = rng.normal(68, 8, n)
vibracao = rng.gamma(2.2, 1.1, n)
corrente = rng.normal(42, 7, n)
pressao = rng.normal(6.2, 0.7, n)

# Rótulo sintético com interação não linear e uma parcela de ruído.
risco = (
    0.09 * (temperatura - 70)
    + 0.65 * (vibracao - 3)
    + 0.05 * (corrente - 45)
    + 0.8 * ((temperatura > 76) & (vibracao > 3.4))
    + rng.normal(0, 0.55, n)
)
falha = (risco > 0.25).astype(int)

df = pd.DataFrame({
    "temperatura_c": temperatura,
    "vibracao_mm_s": vibracao,
    "corrente_a": corrente,
    "pressao_bar": pressao,
    "falha": falha,
})
df.head()

In [ ]:
df.describe().round(2)

## 4. Pipeline comparável e reproduzível

Para uma comparação justa, Random Forest e MLP recebem a mesma divisão estratificada. A MLP recebe atributos padronizados; a floresta não depende dessa padronização, mas permanece no mesmo experimento como baseline.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score
import matplotlib.pyplot as plt

X = df.drop(columns="falha")
y = df["falha"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

mlp = Pipeline([
    ("escala", StandardScaler()),
    ("modelo", MLPClassifier(
        hidden_layer_sizes=(16, 8), activation="relu",
        solver="adam", max_iter=500, early_stopping=True,
        n_iter_no_change=20,
        validation_fraction=0.2, random_state=42
    )),
])

rf = RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")

mlp.fit(X_train, y_train)
rf.fit(X_train, y_train)

for nome, modelo in {"MLP": mlp, "Random Forest": rf}.items():
    pred = modelo.predict(X_test)
    proba = modelo.predict_proba(X_test)[:, 1]
    print(f"\n{nome} — ROC-AUC: {roc_auc_score(y_test, proba):.3f}")
    print(classification_report(y_test, pred, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nome, modelo) in zip(axes, {"MLP": mlp, "Random Forest": rf}.items()):
    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(nome)
plt.tight_layout()

## 5. Lendo o treinamento, não apenas o resultado

Uma métrica final não revela se o modelo convergiu, oscilou ou começou a memorizar. O `MLPClassifier` registra a perda de treinamento e, com `early_stopping=True`, reserva uma parcela do treino para validação interna. O treinamento termina quando essa validação deixa de melhorar pelo número configurado de épocas. O conjunto de teste permanece fora desse processo.

In [ ]:
modelo_mlp = mlp.named_steps["modelo"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(modelo_mlp.loss_curve_, color="#6a1b9a")
axes[0].set(title="Perda de treinamento", xlabel="Época", ylabel="Loss")
axes[0].grid(alpha=0.25)

axes[1].plot(modelo_mlp.validation_scores_, color="#00695c")
axes[1].set(title="Acurácia na validação interna", xlabel="Época", ylabel="Score")
axes[1].grid(alpha=0.25)
plt.tight_layout()
print("Épocas executadas:", modelo_mlp.n_iter_)
print("Melhor score interno:", round(modelo_mlp.best_validation_score_, 4))


## 5. Inspeção da rede treinada

O `MLPClassifier` armazena uma matriz de pesos e um vetor de vieses para cada transição. Para a arquitetura `(4, 16, 8, 1)`, as formas esperadas são `(4,16)`, `(16,8)` e `(8,1)`.


In [ ]:
for i, (pesos, vieses) in enumerate(zip(modelo_mlp.coefs_, modelo_mlp.intercepts_), start=1):
    print(f"Camada {i}: pesos {pesos.shape}, vieses {vieses.shape}, parâmetros {pesos.size + vieses.size}")
print("Total de parâmetros:", sum(w.size + b.size for w, b in zip(modelo_mlp.coefs_, modelo_mlp.intercepts_)))


### Forward pass de uma observação

A célula seguinte aplica o scaler e executa cada multiplicação matricial. A probabilidade manual é comparada à saída oficial de `predict_proba`.


In [ ]:
amostra = X_test.iloc[[0]]
x_scaled = mlp.named_steps["escala"].transform(amostra)

def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

a = x_scaled
ativacoes = [("entrada padronizada", a.copy())]
for camada, (W, b) in enumerate(zip(modelo_mlp.coefs_, modelo_mlp.intercepts_), start=1):
    z = a @ W + b
    a = sigmoid(z) if camada == len(modelo_mlp.coefs_) else relu(z)
    ativacoes.append((f"camada {camada}", a.copy()))

for nome, valores in ativacoes:
    print(nome, "forma=", valores.shape, "valores=", np.round(valores, 4))

proba_manual = float(a.ravel()[0])
proba_sklearn = float(mlp.predict_proba(amostra)[0, 1])
print(f"\nProbabilidade manual: {proba_manual:.8f}")
print(f"predict_proba:       {proba_sklearn:.8f}")
print("Diferença absoluta:", abs(proba_manual - proba_sklearn))
assert np.isclose(proba_manual, proba_sklearn, atol=1e-8)


### Visualização das ativações

O mapa abaixo mostra os valores numéricos produzidos durante o forward pass. Todos os painéis usam a mesma escala de cores, indicada pela barra lateral.

### Interpretação das cores

- **roxo escuro:** menor valor numérico na escala comum;
- **verde:** valor intermediário;
- **amarelo:** maior valor numérico;
- na **entrada padronizada**, valores negativos representam medições abaixo da média do treino e valores positivos representam medições acima da média;
- nas **camadas ocultas**, zero representa um neurônio desativado pela ReLU e cores claras representam ativações positivas mais intensas;
- na **camada de saída**, a cor representa a probabilidade estimada de falha, entre 0 e 1.

> A cor não representa importância da variável ou do neurônio. Ela mostra somente o valor da ativação para esta observação.


In [ ]:
from matplotlib.colors import Normalize

# Uma escala comum torna as cores comparáveis entre os painéis.
todos_valores = np.concatenate([valores.ravel() for _, valores in ativacoes])
norm = Normalize(vmin=todos_valores.min(), vmax=todos_valores.max())

fig, axes = plt.subplots(1, len(ativacoes), figsize=(15, 3.4), constrained_layout=True)
for ax, (nome, valores) in zip(axes, ativacoes):
    imagem = ax.imshow(valores, cmap="viridis", norm=norm, aspect="auto")
    ax.set_title(nome)
    ax.set_yticks([])
    ax.set_xticks(range(valores.shape[1]))
    ax.set_xticklabels(range(1, valores.shape[1] + 1), fontsize=7)
    ax.set_xlabel(f"{valores.shape[1]} unidades")

barra = fig.colorbar(imagem, ax=axes, shrink=0.82, pad=0.02)
barra.set_label("Valor numérico da entrada ou ativação")
plt.show()


## 6. Interpretação orientada ao negócio

Considere uma parada não detectada muito mais cara que uma inspeção desnecessária. Nesse caso, o **recall da classe de falha** pode ser mais importante que a acurácia global. Alterar o limiar de decisão muda o equilíbrio entre falsos positivos e falsos negativos.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = mlp.predict_proba(X_test)[:, 1]
linhas = []
for limiar in [0.30, 0.40, 0.50, 0.60, 0.70]:
    pred = (proba >= limiar).astype(int)
    linhas.append({
        "limiar": limiar,
        "precisão_falha": precision_score(y_test, pred, zero_division=0),
        "recall_falha": recall_score(y_test, pred, zero_division=0),
        "f1_falha": f1_score(y_test, pred, zero_division=0),
    })
pd.DataFrame(linhas).round(3)

In [ ]:
# Cálculo do Impacto Financeiro (Custo Esperado por Limiar)
# Premissa: Falso Negativo (Falha não detectada) = R$ 5.000 | Falso Positivo (Alarme falso) = R$ 500
CUSTO_FN = 5000
CUSTO_FP = 500

linhas_custo = []
for limiar in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = (proba >= limiar).astype(int)
    fn = int(((y_test == 1) & (pred == 0)).sum())
    fp = int(((y_test == 0) & (pred == 1)).sum())
    custo_total = (fn * CUSTO_FN) + (fp * CUSTO_FP)
    linhas_custo.append({
        "limiar": limiar,
        "falsos_negativos": fn,
        "falsos_positivos": fp,
        "recall": round(recall_score(y_test, pred, zero_division=0), 3),
        "precisao": round(precision_score(y_test, pred, zero_division=0), 3),
        "custo_total_rs": custo_total
    })

df_custo = pd.DataFrame(linhas_custo)
print("=== Matriz de Decisão Financeira por Limiar ===")
display(df_custo)
limiar_otimo = df_custo.loc[df_custo['custo_total_rs'].idxmin()]
print(f"\nLimiar Financeiramente Ótimo: {limiar_otimo['limiar']} (Custo Mínimo: R$ {limiar_otimo['custo_total_rs']:,.2f})")


## 7. Atividade de investigação

1. Altere a arquitetura para `(8,)`, `(32, 16)` e `(32, 16, 8)`. Registre o número de épocas e o desempenho.
2. Retire temporariamente o `StandardScaler`. O que muda na estabilidade e no resultado?
3. Compare `relu` e `tanh` mantendo as demais decisões fixas.
4. Escolha um limiar considerando que um falso negativo custa dez vezes mais que um falso positivo.
5. Redija uma conclusão de CRISP-DM: a MLP superou o baseline de maneira suficiente para justificar sua complexidade?

### Evidência esperada

Uma tabela com configuração, métrica técnica, custo estimado e justificativa. Não escolha o modelo apenas pela maior acurácia.

## 8. Síntese

- Redes neurais aprendem transformações sucessivas; árvores combinam regras de particionamento.
- Preparação e escala passam a influenciar diretamente o treinamento.
- A comparação deve usar a mesma divisão e métricas coerentes com o negócio.
- A curva de perda ajuda a diagnosticar a aprendizagem.
- O CRISP-DM continua sendo a estrutura de decisão: o algoritmo é apenas uma peça.

## Bibliografia e aprofundamento

- PEDREGOSA, F. et al. *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research, 2011.
- GÉRON, Aurélien. *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*. 3. ed. O'Reilly, 2022.
- GOODFELLOW, Ian; BENGIO, Yoshua; COURVILLE, Aaron. *Deep Learning*. MIT Press, 2016. Disponível em: <https://www.deeplearningbook.org/>.
- Scikit-learn. **MLPClassifier**. <https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html>.
- Scikit-learn. **StandardScaler**. <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html>.
- Scikit-learn. **Common pitfalls and recommended practices**. <https://scikit-learn.org/stable/common_pitfalls.html>.